<a href="https://colab.research.google.com/github/NarendraRaoJami/Internship_2026_College/blob/main/SparseGPT_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SparseGPT — facebook/opt-125m | wikitext2 | All 3 Methods

In [1]:
!pip install -q transformers==4.36.2 tokenizers==0.15.2 datasets==2.18.0 huggingface_hub==0.23.4 --only-binary=:all:
!pip install -q sentencepiece accelerate
print('Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.4 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.23.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.
gradio 5.50.0 requires huggingfac

In [2]:
import os
if not os.path.exists('sparsegpt'):
    !git clone https://github.com/IST-DASLab/sparsegpt
%cd sparsegpt
!mkdir -p sparse_sparsegpt sparse_magnitude sparse_movement

Cloning into 'sparsegpt'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 46 (delta 22), reused 10 (delta 10), pack-reused 14 (from 2)
Receiving objects: 100% (46/46), 26.80 KiB | 13.40 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/sparsegpt


## Imports & Setup

In [3]:
import torch
import torch.nn.utils.prune as prune
import time
import subprocess
import re
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL     = 'facebook/opt-125m'
DATASET   = 'wikitext2'
SPARSITY_LEVELS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
device    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device : {device}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## Helper — Inference Metrics (Memory, Latency, Throughput, Energy)

In [5]:
def measure_inference(model, tokenizer, num_runs=5):
    model.eval()
    if device == 'cuda':
        torch.cuda.synchronize()
        memory_mb = round(torch.cuda.memory_allocated() / 1e6, 1)
    else:
        import psutil
        memory_mb = round(__import__('psutil').Process(os.getpid()).memory_info().rss / 1e6, 1)

    inputs  = tokenizer('The quick brown fox', return_tensors='pt').to(device)
    MAX_NEW = 50

    with torch.no_grad():
        _ = model.generate(inputs['input_ids'], max_new_tokens=5, do_sample=False)

    latencies = []
    for _ in range(num_runs):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            _ = model.generate(inputs['input_ids'], max_new_tokens=MAX_NEW, do_sample=False)
        if device == 'cuda': torch.cuda.synchronize()
        latencies.append(time.time() - t0)

    avg_lat    = float(np.mean(latencies))
    throughput = round(MAX_NEW / avg_lat, 2)
    tdp_w      = 400 if 'a100' in torch.cuda.get_device_name(0).lower() else 70
    energy_j   = round(tdp_w * avg_lat, 3)

    return {
        'memory_mb':  memory_mb,
        'latency_s':  round(avg_lat, 3),
        'throughput': throughput,
        'energy_j':   energy_j,
    }

## Method 1 — SparseGPT

Uses the official `opt.py` script. Perplexity is taken directly from script output.

In [6]:
results_sparsegpt = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_sparsegpt/s{spct}'

    print(f'\nSparseGPT {spct}%')

    t0     = time.time()
    result = subprocess.run(
        ['python', 'opt.py', MODEL, DATASET,
         '--sparsity', str(sparsity), '--save', save_path],
        capture_output=True, text=True
    )
    pruning_time = round(time.time() - t0, 1)

    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    save_path, torch_dtype=torch.float16).to(device)
    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    results_sparsegpt[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nSparseGPT done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_sparsegpt.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')


SparseGPT 20%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_

  PPL=28.17617 | Mem=257.4MB | Lat=0.254s | Tok/s=196.79 | PruneTime=256.4s | Energy=17.785J

SparseGPT 30%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=28.922119 | Mem=267.9MB | Lat=0.145s | Tok/s=345.16 | PruneTime=229.6s | Energy=10.14J

SparseGPT 40%


...

  PPL=33.194546 | Mem=267.9MB | Lat=0.648s | Tok/s=77.12 | PruneTime=236.7s | Energy=45.383J

SparseGPT 60%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=46.305183 | Mem=267.9MB | Lat=0.57s | Tok/s=87.73 | PruneTime=226.4s | Energy=39.894J

SparseGPT 70%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=134.661255 | Mem=267.9MB | Lat=0.518s | Tok/s=96.52 | PruneTime=257.5s | Energy=36.26J

SparseGPT done

  Sparsity |      PPL |  Mem(MB) |  Lat(s) |   Tok/s |  PruneTime |  Energy(J)
---------------------------------------------------------------------------
       20% | 28.17617 |    257.4 |   0.254 |  196.79 |      256.4 |     17.785
       30% | 28.922119 |    267.9 |   0.145 |  345.16 |      229.6 |      10.14
       40% | 30.215551 |    267.9 |   0.542 |   92.24 |      231.4 |     37.946
       50% | 33.194546 |    267.9 |   0.648 |   77.12 |      236.7 |     45.383
       60% | 46.305183 |    267.9 |    0.57 |   87.73 |      226.4 |     39.894
       70% | 134.661255 |    267.9 |   0.518 |   96.52 |      257.5 |      36.26


## Method 2 — Magnitude Pruning

Uses `torch.nn.utils.prune.l1_unstructured`. Perplexity computed via `opt.py` by saving and re-evaluating the pruned model.

In [7]:
results_magnitude = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_magnitude/s{spct}'
    os.makedirs(save_path, exist_ok=True)

    print(f'\nMagnitude {spct}%')

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL, torch_dtype=torch.float16).to(device)

    t0 = time.time()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=sparsity)
            prune.remove(module, 'weight')
    pruning_time = round(time.time() - t0, 1)

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    result = subprocess.run(
        ['python', 'opt.py', save_path, DATASET, '--sparsity', '0'],
        capture_output=True, text=True
    )
    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    results_magnitude[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nMagnitude done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_magnitude.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')


Magnitude 20%
  PPL=29.88228 | Mem=267.9MB | Lat=0.26s | Tok/s=192.3 | PruneTime=0.1s | Energy=18.201J

Magnitude 30%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=36.086674 | Mem=267.9MB | Lat=0.309s | Tok/s=161.97 | PruneTime=0.1s | Energy=21.609J

Magnitude 40%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=65.299858 | Mem=267.9MB | Lat=0.512s | Tok/s=97.65 | PruneTime=0.1s | Energy=35.844J

Magnitude 50%


...

  PPL=376.940369 | Mem=267.9MB | Lat=0.527s | Tok/s=94.94 | PruneTime=0.1s | Energy=36.867J

Magnitude 60%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=2509.167725 | Mem=267.9MB | Lat=0.516s | Tok/s=96.87 | PruneTime=0.1s | Energy=36.131J

Magnitude 70%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=7155.73584 | Mem=267.9MB | Lat=0.524s | Tok/s=95.45 | PruneTime=0.2s | Energy=36.67J

Magnitude done

  Sparsity |      PPL |  Mem(MB) |  Lat(s) |   Tok/s |  PruneTime |  Energy(J)
---------------------------------------------------------------------------
       20% | 29.88228 |    267.9 |    0.26 |   192.3 |        0.1 |     18.201
       30% | 36.086674 |    267.9 |   0.309 |  161.97 |        0.1 |     21.609
       40% | 65.299858 |    267.9 |   0.512 |   97.65 |        0.1 |     35.844
       50% | 376.940369 |    267.9 |   0.527 |   94.94 |        0.1 |     36.867
       60% | 2509.167725 |    267.9 |   0.516 |   96.87 |        0.1 |     36.131
       70% | 7155.73584 |    267.9 |   0.524 |   95.45 |        0.2 |      36.67


## Method 3 — Movement Pruning

Scores weights by `|weight × gradient|` using a calibration pass. Saves and evaluates with `opt.py`.

In [8]:
results_movement = {}

for sparsity in SPARSITY_LEVELS:
    spct      = int(sparsity * 100)
    save_path = f'sparse_movement/s{spct}'
    os.makedirs(save_path, exist_ok=True)

    print(f'\nMovement {spct}%')

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model     = AutoModelForCausalLM.from_pretrained(
                    MODEL, torch_dtype=torch.float32).to(device)
    model.train()

    scores = {n: torch.zeros_like(m.weight.data)
              for n, m in model.named_modules()
              if isinstance(m, torch.nn.Linear)}

    calib = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    texts = [x for x in calib['text'] if len(x.strip()) > 50][:16]

    t0 = time.time()
    for text in texts:
        enc = tokenizer(text, return_tensors='pt',
                        truncation=True, max_length=128).to(device)
        out = model(**enc, labels=enc['input_ids'])
        out.loss.backward()
        for n, m in model.named_modules():
            if isinstance(m, torch.nn.Linear) and m.weight.grad is not None:
                scores[n] += (m.weight.data * m.weight.grad).abs()
        model.zero_grad()

    model.eval()
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            k      = int(sparsity * scores[n].numel())
            thresh = scores[n].flatten().kthvalue(k).values
            m.weight.data *= (scores[n] > thresh).float()
    pruning_time = round(time.time() - t0, 1)

    model = model.half()
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    inf = measure_inference(model, tokenizer)
    del model
    torch.cuda.empty_cache()

    result = subprocess.run(
        ['python', 'opt.py', save_path, DATASET, '--sparsity', '0'],
        capture_output=True, text=True
    )
    output = result.stdout + result.stderr
    match  = re.search(r'wikitext2.*?Perplexity:\s*([\d.]+)', output,
                       re.IGNORECASE | re.DOTALL)
    if not match:
        matches = re.findall(r'Perplexity:\s*([\d.]+)', output)
        ppl = float(matches[0]) if matches else None
    else:
        ppl = float(match.group(1))

    results_movement[spct] = {
        'sparsity':      f'{spct}%',
        'perplexity':    ppl,
        'memory_mb':     inf['memory_mb'],
        'latency_s':     inf['latency_s'],
        'pruning_time':  pruning_time,
        'energy_j':      inf['energy_j'],
        'throughput':    inf['throughput'],
    }

    print(f'  PPL={ppl} | Mem={inf["memory_mb"]}MB | Lat={inf["latency_s"]}s | '
          f'Tok/s={inf["throughput"]} | PruneTime={pruning_time}s | Energy={inf["energy_j"]}J')

print('\nMovement done')
print(f'\n{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
print('-'*75)
for s, r in results_movement.items():
    print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | {str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | {str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')


Movement 20%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=30.202902 | Mem=904.0MB | Lat=0.182s | Tok/s=274.15 | PruneTime=1.2s | Energy=12.767J

Movement 30%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=40.594288 | Mem=904.4MB | Lat=0.57s | Tok/s=87.66 | PruneTime=0.9s | Energy=39.929J

Movement 40%


...

  PPL=251.552277 | Mem=906.0MB | Lat=0.545s | Tok/s=91.69 | PruneTime=0.9s | Energy=38.171J

Movement 60%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=802.986023 | Mem=904.9MB | Lat=0.547s | Tok/s=91.38 | PruneTime=0.9s | Energy=38.302J

Movement 70%


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  PPL=1697.50354 | Mem=906.9MB | Lat=0.573s | Tok/s=87.24 | PruneTime=1.0s | Energy=40.118J

Movement done

  Sparsity |      PPL |  Mem(MB) |  Lat(s) |   Tok/s |  PruneTime |  Energy(J)
---------------------------------------------------------------------------
       20% | 30.202902 |    904.0 |   0.182 |  274.15 |        1.2 |     12.767
       30% | 40.594288 |    904.4 |    0.57 |   87.66 |        0.9 |     39.929
       40% | 184.422531 |    905.8 |   0.565 |   88.57 |        0.9 |     39.515
       50% | 251.552277 |    906.0 |   0.545 |   91.69 |        0.9 |     38.171
       60% | 802.986023 |    904.9 |   0.547 |   91.38 |        0.9 |     38.302
       70% | 1697.50354 |    906.9 |   0.573 |   87.24 |        1.0 |     40.118


## Final Summary — All 3 Methods

In [9]:
all_results = {
    'SparseGPT': results_sparsegpt,
    'Magnitude': results_magnitude,
    'Movement':  results_movement,
}

for method, res in all_results.items():
    print(f'\n=== {method} ===')
    print(f'{"Sparsity":>10} | {"PPL":>8} | {"Mem(MB)":>8} | {"Lat(s)":>7} | {"Tok/s":>7} | {"PruneTime":>10} | {"Energy(J)":>10}')
    print('-'*75)
    for s, r in res.items():
        print(f'{str(s)+"%":>10} | {str(r["perplexity"]):>8} | {str(r["memory_mb"]):>8} | '
              f'{str(r["latency_s"]):>7} | {str(r["throughput"]):>7} | '
              f'{str(r["pruning_time"]):>10} | {str(r["energy_j"]):>10}')

print('\n=== Perplexity Comparison (wikitext2) ===')
print(f'{"Sparsity":>10} | {"SparseGPT":>11} | {"Magnitude":>11} | {"Movement":>11}')
print('-'*50)
for s in [20,30,40,50,60,70]:
    sg = results_sparsegpt.get(s,{}).get('perplexity','N/A')
    mg = results_magnitude.get(s,{}).get('perplexity','N/A')
    mv = results_movement.get(s,{}).get('perplexity','N/A')
    print(f'{str(s)+"%":>10} | {str(sg):>11} | {str(mg):>11} | {str(mv):>11}')


=== SparseGPT ===
  Sparsity |      PPL |  Mem(MB) |  Lat(s) |   Tok/s |  PruneTime |  Energy(J)
---------------------------------------------------------------------------
       20% | 28.17617 |    257.4 |   0.254 |  196.79 |      256.4 |     17.785
       30% | 28.922119 |    267.9 |   0.145 |  345.16 |      229.6 |      10.14
       40% | 30.215551 |    267.9 |   0.542 |   92.24 |      231.4 |     37.946
       50% | 33.194546 |    267.9 |   0.648 |   77.12 |      236.7 |     45.383
       60% | 46.305183 |    267.9 |    0.57 |   87.73 |      226.4 |     39.894
       70% | 134.661255 |    267.9 |   0.518 |   96.52 |      257.5 |      36.26

=== Magnitude ===
  Sparsity |      PPL |  Mem(MB) |  Lat(s) |   Tok/s |  PruneTime |  Energy(J)
---------------------------------------------------------------------------
       20% | 29.88228 |    267.9 |    0.26 |   192.3 |        0.1 |     18.201
       30% | 36.086674 |    267.9 |   0.309 |  161.97 |        0.1 |     21.609
       40% | 